In [1]:
%load_ext bigquery_magics

## 1. QUALIFY句とは

- MySQLなどのRDBMSでは、ウィンドウ関数の結果を`WHERE`句で直接絞り込むことはできない。

- `WHERE`句はウィンドウ関数の計算より先に処理されるため、インラインビューまたはCTEを使用する必要がある。

- BigQueryでは、`QUALIFY`句を使用するとウィンドウ関数の結果を直接絞り込める。

In [ ]:
%%bigquery

select product ,amount, rank() over(order by amount) r
from test.sales
where r = 5; -- ERROR

Executing query with job ID: c71a09d4-a590-4f14-b019-8e9e2cd9d44c
Query executing: 1.24s


ERROR:
 400 Unrecognized name: r at [3:7]; reason: invalidQuery, location: query, message: Unrecognized name: r at [3:7]

Location: US
Job ID: c71a09d4-a590-4f14-b019-8e9e2cd9d44c



#### 1）QUALIFY句を使用

実行順序： `FROM → WHERE → SELECT → QUALIFY`

In [4]:
%%bigquery

select product ,amount, rank() over(order by amount) r
from `test.sales`
qualify r = 5;

Query is running:   0%|          |

Downloading:   0%|          |

,product,amount,r
0,모니터,50,5


#### 2）インラインビューを使用

In [5]:
%%bigquery

SELECT
    product,
    amount,
    r
FROM (
    SELECT
        product,
        amount,
        RANK() OVER (ORDER BY amount) AS r
    FROM `test.sales`
)
WHERE r = 5;

Query is running:   0%|          |

Downloading:   0%|          |

,product,amount,r
0,모니터,50,5


#### 3）CTEを使用

In [6]:
%%bigquery

WITH ranked AS (
    SELECT
        product,
        amount,
        RANK() OVER (ORDER BY amount) AS r
    FROM `test.sales`
)
SELECT
    product,
    amount,
    r
FROM ranked
WHERE r = 5;

Query is running:   0%|          |

Downloading:   0%|          |

,product,amount,r
0,모니터,50,5
